In [1]:

!nvidia-smi
from numba import cuda
print(cuda.gpus)
dev = cuda.get_current_device()
print("GPU:", dev.name)
print("Compute capability:", dev.compute_capability)

Thu Oct  8 10:02:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
%%writefile task1_divergence.py
"""Task 1: Warp divergence microbenchmark."""
import time
import numpy as np
from numba import cuda

ITERS = 1000
N = 1 << 22  # 4,194,304 elements (multiple of 32, so warps are full)
TPB = 256


@cuda.jit
def kernel_uniform(arr, n):
    idx = cuda.grid(1)
    if idx < n:
        acc = arr[idx]
        for _ in range(ITERS):
            acc = acc * 1.0001 + 0.5  # multiply-accumulate
        arr[idx] = acc


@cuda.jit
def kernel_divergent(arr, n):
    idx = cuda.grid(1)
    if idx < n:
        acc = arr[idx]
        if idx % 2 == 0:
            for _ in range(ITERS):
                acc = acc * 1.0001 + 0.5  # Path 1: MAC
        else:
            for _ in range(ITERS):
                acc = (acc - 0.5) / 1.0001  # Path 2: subtract-divide
        arr[idx] = acc


@cuda.jit
def kernel_warp_aligned(arr, n):
    idx = cuda.grid(1)
    if idx < n:
        acc = arr[idx]
        warp_id = idx // 32
        if warp_id % 2 == 0:
            for _ in range(ITERS):
                acc = acc * 1.0001 + 0.5  # Path 1: MAC
        else:
            for _ in range(ITERS):
                acc = (acc - 0.5) / 1.0001  # Path 2: subtract-divide
        arr[idx] = acc


def bench(kernel, d_arr, n, trials=10):
    blocks = (n + TPB - 1) // TPB
    kernel[blocks, TPB](d_arr, n)  # warm-up (also triggers JIT compile)
    cuda.synchronize()
    times = []
    for _ in range(trials):
        cuda.synchronize()
        t0 = time.perf_counter()
        kernel[blocks, TPB](d_arr, n)
        cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000.0)
    return sum(times) / len(times)


def main():
    h = np.random.rand(N).astype(np.float32)
    d = cuda.to_device(h)  # transfer excluded from timing
    t_a = bench(kernel_uniform, d, N)
    t_b = bench(kernel_divergent, d, N)
    t_c = bench(kernel_warp_aligned, d, N)

    print("| Kernel | Avg time (ms) | Slowdown vs A |")
    print("|---|--:|--:|")
    print(f"| A: Uniform path | {t_a:.3f} | 1.00x |")
    print(f"| B: Full divergence (idx % 2) | {t_b:.3f} | {t_b / t_a:.2f}x |")
    print(f"| C: Warp-aligned (warp_id % 2) | {t_c:.3f} | {t_c / t_a:.2f}x |")


if __name__ == "__main__":
    main()


Overwriting task1_divergence.py


In [6]:
%%writefile task2_stencil_1d.py
"""Task 2: 1D 3-point smoothing stencil with boundary clamping."""
import numpy as np
from numba import cuda

TPB = 256


@cuda.jit
def stencil_1d(d_in, d_out, N):
    idx = cuda.grid(1)
    if idx < N:
        left = d_in[idx - 1] if idx > 0 else d_in[0]          # clamp left
        right = d_in[idx + 1] if idx < N - 1 else d_in[N - 1]  # clamp right
        d_out[idx] = 0.25 * left + 0.5 * d_in[idx] + 0.25 * right


def run_stencil(h_in):
    h_in = np.ascontiguousarray(h_in, dtype=np.float32)
    N = h_in.size
    d_in = cuda.to_device(h_in)
    d_out = cuda.device_array(N, dtype=np.float32)
    blocks = (N + TPB - 1) // TPB
    stencil_1d[blocks, TPB](d_in, d_out, N)
    cuda.synchronize()
    return d_out.copy_to_host()


def cpu_stencil(arr):
    padded = np.pad(arr, (1, 1), mode='edge')
    return 0.25 * padded[:-2] + 0.5 * padded[1:-1] + 0.25 * padded[2:]


if __name__ == "__main__":
    N = 10007  # odd, non-power-of-two
    h_in = np.sin(np.linspace(0, 10, N)).astype(np.float32)
    h_out_gpu = run_stencil(h_in)
    cpu_ref = cpu_stencil(h_in)
    assert np.allclose(h_out_gpu, cpu_ref, atol=1e-4)
    delta = float(np.max(np.abs(h_out_gpu - cpu_ref)))
    print(f"TASK 2 PASSED: MAX DELTA = {delta}")


Overwriting task2_stencil_1d.py


In [7]:
%%writefile task3_grid_stride.py
"""Task 3: Grid-stride loop scaling with a fixed hardware grid."""
import numpy as np
from numba import cuda

THREADS_PER_BLOCK = 256
BLOCKS_PER_GRID = 64  # 256 * 64 = 16,384 hardware threads, no more


@cuda.jit
def grid_stride_scale_kernel(d_arr, factor, N):
    start = cuda.grid(1)
    stride = cuda.gridsize(1)
    for i in range(start, N, stride):
        d_arr[i] = d_arr[i] * factor


def run_grid_stride(h_arr, factor):
    h_arr = np.ascontiguousarray(h_arr, dtype=np.float32)
    d_arr = cuda.to_device(h_arr)
    grid_stride_scale_kernel[BLOCKS_PER_GRID, THREADS_PER_BLOCK](
        d_arr, np.float32(factor), h_arr.size)
    cuda.synchronize()
    return d_arr.copy_to_host()


if __name__ == "__main__":
    N = 1_000_003  # far larger than 16,384 threads, not a multiple of the stride
    factor = 4.25
    res = run_grid_stride(np.ones(N, dtype=np.float32), factor)
    assert np.allclose(res, factor)
    print(f"TASK 3 PASSED: {N} elements, all equal {factor}")


Writing task3_grid_stride.py


In [8]:
%%writefile task4_sobel_2d.py
"""Task 4: 2D Sobel-X filter."""
import numpy as np
from numba import cuda

THREADS_2D = (16, 16)


@cuda.jit
def sobel_x_kernel(d_in, d_out, rows, cols):
    col, row = cuda.grid(2)
    if row < rows and col < cols:
        if 0 < row < rows - 1 and 0 < col < cols - 1:
            right = (d_in[row - 1, col + 1]
                     + 2.0 * d_in[row, col + 1]
                     + d_in[row + 1, col + 1])
            left = (d_in[row - 1, col - 1]
                    + 2.0 * d_in[row, col - 1]
                    + d_in[row + 1, col - 1])
            d_out[row, col] = right - left
        else:
            d_out[row, col] = 0.0  # border pixels


def run_sobel(h_img):
    h_img = np.ascontiguousarray(h_img, dtype=np.float32)
    rows, cols = h_img.shape
    d_in = cuda.to_device(h_img)
    d_out = cuda.device_array((rows, cols), dtype=np.float32)
    blocks = ((cols + THREADS_2D[0] - 1) // THREADS_2D[0],
              (rows + THREADS_2D[1] - 1) // THREADS_2D[1])
    sobel_x_kernel[blocks, THREADS_2D](d_in, d_out, rows, cols)
    cuda.synchronize()
    return d_out.copy_to_host()


def cpu_sobel(img):
    out = np.zeros_like(img)
    out[1:-1, 1:-1] = (
        (img[:-2, 2:] + 2 * img[1:-1, 2:] + img[2:, 2:])
        - (img[:-2, :-2] + 2 * img[1:-1, :-2] + img[2:, :-2]))
    return out


if __name__ == "__main__":
    rows, cols = 1080, 1920  # not multiples of 16, tests the guards
    img = np.random.rand(rows, cols).astype(np.float32)
    gpu = run_sobel(img)
    assert np.allclose(gpu, cpu_sobel(img), atol=1e-4)
    print(f"TASK 4 PASSED: {rows}x{cols} matches CPU reference")


Writing task4_sobel_2d.py


In [9]:
%%writefile verify_submission.py
"""
AUTONOMOUS VERIFICATION & INTEGRITY TOKEN GENERATOR
Run: python verify_submission.py
"""
import sys
import hashlib
import numpy as np
from numba import cuda


def main():
    print("=" * 60)
    print("RUNNING CUDA LAB 02 AUTONOMOUS VERIFICATION")
    print("=" * 60)
    student_id = input("Enter your Student ID: ").strip()
    if not student_id:
        print("[FAIL] Student ID cannot be empty.")
        sys.exit(1)

    # 1. Verify Task 2
    try:
        import task2_stencil_1d as t2
        N = 10007
        test_in = np.sin(np.linspace(0, 10, N)).astype(np.float32)
        h_gpu = t2.run_stencil(test_in)
        h_cpu = t2.cpu_stencil(test_in)
        assert np.allclose(h_gpu, h_cpu, atol=1e-4), "Task 2 output mismatch against CPU"
        print("[PASS] Task 2 (1D Stencil & Clamping)")
    except Exception as e:
        print(f"[FAIL] Task 2: {e}")
        sys.exit(1)

    # 2. Verify Task 3
    try:
        import task3_grid_stride as t3
        N = 100000
        test_arr = np.ones(N, dtype=np.float32)
        factor = 4.25
        res = t3.run_grid_stride(test_arr, factor)
        assert np.allclose(res, factor), "Task 3 elements not uniformly scaled"
        print("[PASS] Task 3 (Grid-Stride Scaling)")
    except Exception as e:
        print(f"[FAIL] Task 3: {e}")
        sys.exit(1)

    # 3. Verify Task 4
    try:
        import task4_sobel_2d as t4
        test_img = np.ones((64, 64), dtype=np.float32)
        sobel_res = t4.run_sobel(test_img)
        assert np.max(np.abs(sobel_res[1:-1, 1:-1])) < 1e-5, "Task 4 interior gradient != 0 for flat field"
        assert np.all(sobel_res[0, :] == 0.0), "Task 4 border rows not zeroed"
        assert np.all(sobel_res[:, 0] == 0.0), "Task 4 border columns not zeroed"
        print("[PASS] Task 4 (2D Sobel Horizontal)")
    except Exception as e:
        print(f"[FAIL] Task 4: {e}")
        sys.exit(1)

    # Cryptographic Checksum Token Generation
    hasher = hashlib.sha256()
    hasher.update(student_id.encode('utf-8'))
    try:
        device_name = cuda.get_current_device().name
        hasher.update(device_name if isinstance(device_name, bytes) else device_name.encode('utf-8'))
    except Exception:
        hasher.update(b"UNKNOWN_CUDA_DEVICE")
    hasher.update(h_gpu[:32].tobytes())
    hasher.update(sobel_res[:8, :8].tobytes())
    token = hasher.hexdigest()[:20].upper()

    print("\n" + "=" * 60)
    print("VERIFICATION SUCCESSFUL")
    print(f"OFFICIAL SUBMISSION TOKEN: {token}")
    print("=" * 60)
    print("Copy this token directly into your README.md.\n")


if __name__ == "__main__":
    main()


Writing verify_submission.py


In [10]:
!python task1_divergence.py

| Kernel | Avg time (ms) | Slowdown vs A |
|---|--:|--:|
| A: Uniform path | 47.420 | 1.00x |
| B: Full divergence (idx % 2) | 372.243 | 7.85x |
| C: Warp-aligned (warp_id % 2) | 186.282 | 3.93x |


In [11]:
!python task2_stencil_1d.py

/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 40 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
TASK 2 PASSED: MAX DELTA = 5.960464477539063e-08


In [12]:
!python task3_grid_stride.py

/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 64 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
TASK 3 PASSED: 1000003 elements, all equal 4.25


In [13]:
!python task4_sobel_2d.py

TASK 4 PASSED: 1080x1920 matches CPU reference


In [14]:
STUDENT_ID = "230103201"
!echo {STUDENT_ID} | python verify_submission.py

RUNNING CUDA LAB 02 AUTONOMOUS VERIFICATION
Enter your Student ID: /usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 40 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
[PASS] Task 2 (1D Stencil & Clamping)
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 64 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
[PASS] Task 3 (Grid-Stride Scaling)
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 16 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
[PASS] Task 4 (2D Sobel Horizontal)

VERIFICATION SUCCESSFUL
OFFICIAL SUBMISSION TOKEN: A437E097A2A3A76C637C
Copy this token directly into your README.md.



SyntaxError: invalid decimal literal (2279320182.py, line 15)

# CUDA Lab 02: Advanced Geometries & Stencils

**Student ID:** 230103201
**Allocated GPU Node:** Tesla T4
**CUDA Compute Capability:** 7.5
**Official Verification Token:** A437E097A2A3A76C637C

## Task 1: Warp Divergence Benchmark

Array size: 4,194,304 float32, 1,000 iterations per element, 256 threads per block,
kernel-only timing (no host/device transfers), 1 warm-up launch, mean of 10 trials.

| Kernel | Avg time (ms) | Slowdown vs A |
|---|--:|--:|
| A: Uniform path | 47.420 | 1.00x |
| B: Full divergence (idx % 2) | 372.243 | 7.85x |
| C: Warp-aligned (warp_id % 2) | 186.282 | 3.93x |

**Analysis:**

Kernel B is the slowest because even and odd threads share every warp. The warp runs the
multiply-accumulate path with the odd lanes masked off, then the subtract-divide path
with the even lanes masked off, so its time is about the sum of both paths: 47.4 ms for
path 1 (the same work as Kernel A) plus about 325 ms for path 2, or about 372 ms. That
matches the measured value.

Kernel C is 3.93x slower than A, but divergence does not explain that gap. The
subtract-divide path costs about 7x more than the multiply-accumulate path, since float
division compiles to several instructions and each iteration depends on the previous one.
In C every warp takes a single branch, so half the warps do about 47 ms of work and the
other half do about 325 ms. The average is about 186 ms, which matches the measurement.

The right comparison for divergence is B against C, because both run the same two paths
on the same data. B takes 372.2 ms and C takes 186.3 ms, so intra-warp divergence about
doubles the runtime. Every warp in B pays for both paths, and every warp in C pays for one.

## Task 2: 1D Stencil

TASK 2 PASSED: MAX DELTA = 5.960464477539063e-08 (N = 10007, atol = 1e-4)

## Task 3: Grid-Stride Scaling

Launch: 64 blocks x 256 threads = 16,384 threads for N = 1,000,003 elements.
TASK 3 PASSED: 1000003 elements, all equal 4.25

## Task 4: Sobel-X

Launch: (16, 16) threads per block, grid = ceil(cols/16) x ceil(rows/16).
TASK 4 PASSED: 1080x1920 matches CPU reference

## Notes

Numba printed a NumbaPerformanceWarning about low occupancy for Tasks 2-4. This is
expected: Task 3 uses a fixed grid of 64 blocks as the lab requires, and the grids in the
other tasks are small because the test inputs are small.